In [10]:
import networkx as nx
import pandas as pd
import numpy as np

In [17]:
G = nx.read_gml("./network_graph_weighted.gml") #TODO fix path 


In [ ]:
# Create a temporary mapping of lowercase keys to original keys
lowercase_mapping = {key.lower(): key for key in G.nodes}

# Query using a case-insensitive key
query_key = "l861q_EGFR".lower()
original_key = lowercase_mapping.get(query_key)

if original_key:
    result = G[original_key]
    print(result)
else:
    print(f"Node '{query_key}' not found in the graph.")

KeyError: 'l861q_egfr'

In [12]:
user_input_cancer = "NSCLC"
variant_of_interest = 'l858r_EGFR' #as durggable usecase
cancer_alias_map = {
    "nsclc": "lung cancer",
    "non-small cell lung cancer": "lung cancer",
    "tnbc": "breast cancer",
    "her2+ breast cancer": "breast cancer"
}
clean_input = user_input_cancer.strip().lower()
cancer_of_interest = cancer_alias_map.get(clean_input, clean_input)
display_cancer_name = user_input_cancer  # Always show the original user input
print(f"\n\n\033[1mCancer of interest set to:\033[0m {display_cancer_name} (cancer type:'{cancer_of_interest}')")
print(f"\033[1mVariant of interest set to:\033[0m {variant_of_interest}")



Cancer of interest set to: NSCLC (cancer type:'lung cancer')
Variant of interest set to: l858r_EGFR


In [ ]:
# === Adjustable thresholds ===
TREATMENT_THRESHOLD_PERCENTILE = 75    # highlight top X% of treatment weights
TREATMENT_MIN_HIGHLIGHT        = 300   # and require ≥X total weight
CANCER_THRESHOLD_PERCENTILE    = 75    # highlight top X% of cancer–variant weights
CANCER_MIN_HIGHLIGHT           = 80    # and require ≥X total weight

# === Prepare consensus lookup ===
df_consensus["Variant_Treatment_Pair"] = (
    df_consensus["Variant_Treatment_Pair"]
    .str.strip()
    .str.lower()
)
consensus_dict = dict(
    zip(df_consensus["Variant_Treatment_Pair"], df_consensus["Resolved_Prediction"])
)

excluded_treatments = {
    'chemotherapy', 'tyrosine kinase inhibitor', 'radiotherapy', 'hormone therapy',
    'adjuvant chemotherapy', 'immunotherapy', 'immune checkpoint inhibitor',
    'mrna vaccine', 'mtor inhibitor', 'radiation ionizing radiotherapy'
}

# === Step 1: Cancer‐only treatments ===
canc_nei = set(G.neighbors(cancer_of_interest))
treatments = [
    n for n in canc_nei
    if G.nodes[n]['category']=='Treatment'
    and n.lower() not in excluded_treatments
]
t_weights = {t: G[cancer_of_interest][t]['weight'] for t in treatments}
top_cancer_treats = sorted(t_weights.items(), key=lambda x: x[1], reverse=True)[:6]
c_w = list(t_weights.values())
treat_pct = np.percentile(c_w, TREATMENT_THRESHOLD_PERCENTILE) if c_w else 0


# === Step 2: Variant + cancer associations ===
sensitive, resistant = [], []
for t in treatments:
    try:
        w = G[cancer_of_interest][t]['weight'] + G[variant_of_interest][t]['weight']
        pred = consensus_dict.get(f"{variant_of_interest} + {t}".lower())
        if pred == "Sensitive":
            sensitive.append((t, w))
        elif pred == "Resistant":
            resistant.append((t, w))
    except KeyError:
        continue

top_sens = sorted(sensitive, key=lambda x: x[1], reverse=True)[:6]
top_res  = sorted(resistant, key=lambda x: x[1], reverse=True)[:6]
sens_w = [w for _, w in sensitive]
res_w  = [w for _, w in resistant]
sens_pct = np.percentile(sens_w, TREATMENT_THRESHOLD_PERCENTILE) if sens_w else 0
res_pct  = np.percentile(res_w,   TREATMENT_THRESHOLD_PERCENTILE) if res_w else 0

print(f"\n\033[1mSensitive treatments for variant '{variant_of_interest}' "
      f"(≥{TREATMENT_THRESHOLD_PERCENTILE}th pct & ≥{TREATMENT_MIN_HIGHLIGHT}):\033[0m")
for t, w in top_sens:
    if w >= sens_pct and w >= TREATMENT_MIN_HIGHLIGHT:
        print(f"\033[1;32m{t}: {w:.0f}\033[0m")
    else:
        print(f"\033[2;37m{t}: {w:.0f}\033[0m")

print(f"\n\033[1mResistant treatments for variant '{variant_of_interest}' "
      f"(≥{TREATMENT_THRESHOLD_PERCENTILE}th pct & ≥{TREATMENT_MIN_HIGHLIGHT}):\033[0m")
for t, w in top_res:
    if w >= res_pct and w >= TREATMENT_MIN_HIGHLIGHT:
        print(f"\033[1;31m{t}: {w:.0f}\033[0m")
    else:
        print(f"\033[2;37m{t}: {w:.0f}\033[0m")

# === Step 3: Other cancers for variant ===
var_nei = set(G.neighbors(variant_of_interest))
var_cancers = [
    n for n in var_nei
    if G.nodes[n]['category']=='Cancer'
    and n != cancer_of_interest
]
vc_weights = {}
for c in var_cancers:
    w_v = G[variant_of_interest][c]['weight']
    w_c = G[cancer_of_interest][c]['weight'] if G.has_edge(cancer_of_interest, c) else 0
    vc_weights[c] = w_v + w_c

top_var_c = sorted(vc_weights.items(), key=lambda x: x[1], reverse=True)[:6]
vc_w = list(vc_weights.values())
cancer_pct = np.percentile(vc_w, CANCER_THRESHOLD_PERCENTILE) if vc_w else 0

print(f"\n\033[1mOther cancers for variant '{variant_of_interest}' "
      f"(≥{CANCER_THRESHOLD_PERCENTILE}th pct & ≥{CANCER_MIN_HIGHLIGHT}):\033[0m")
for c, w in top_var_c:
    if w >= cancer_pct and w >= CANCER_MIN_HIGHLIGHT:
        print(f"\033[1;34m{c}: {w:.0f}\033[0m")
    else:
        print(f"\033[2;37m{c}: {w:.0f}\033[0m")


Sensitive treatments for variant 'l858r_EGFR' (≥75th pct & ≥300):
Osimertinib: 678
Gefitinib: 474
Erlotinib: 432
Afatinib: 333
Radiation Therapy: 167
Crizotinib: 160

Resistant treatments for variant 'l858r_EGFR' (≥75th pct & ≥300):
Cisplatin: 180
Pembrolizumab: 41
Paclitaxel: 30
Docetaxel: 30
Nivolumab: 21
Lapatinib: 17

Other cancers for variant 'l858r_EGFR' (≥75th pct & ≥80):
colon cancer: 92
breast cancer: 64
squamous cell cancer: 60
melanoma: 56
pancreatic cancer: 41
thyroid cancer: 30
